# PE-SC Project Part 1

Fixes applied vs. the original notebook:
- The Implication model now sees **both** X1 and X2, not just X1.
- Evaluation metrics are computed on the model's actual predictions, not the scaled X test values.
- Separate `MinMaxScaler` instances are used for X and y instead of one shared, repeatedly-refit scaler.
- The Implication and XOR pipelines now use distinctly-named variables (`impl_*` / `xor_*`) instead of both reusing `scaler` / `mlp_grid`, since the original silently overwrote the Implication model with the XOR model by the time the notebook finished running.
- Added classification-style metrics (accuracy/F1) alongside RMSE/R2, since `Target` is actually binary.
- The five near-identical "smallest two rows" cells are now one small function.
- Paths no longer assume Google Colab's `/content/...`.
- Added a manual-input cell and a no-learning classical baseline for each model, so you can try your own (X1, X2) values and see whether the trained network is actually adding anything over just computing the rule.
- Added a "What this shows" note after every result-producing cell, so the notebook reads as a self-contained writeup rather than just code and numbers.
- Added a final section training on the true continuous Zadeh fuzzy operators (min/max, not threshold-at-0.5 Boolean logic), since everything above this point was really learning a crisp rule dressed up in continuous inputs.

In [ ]:
import os
import zipfile

DATA_DIR = "."
zip_file_path = "archive.zip"

if os.path.exists(zip_file_path):
    with zipfile.ZipFile(zip_file_path, "r") as zip_ref:
        zip_ref.extractall(DATA_DIR)

print(f"Using data directory: {os.path.abspath(DATA_DIR)}")

In [ ]:
extracted_files = [f for f in os.listdir(DATA_DIR) if f.lower().endswith(".xlsx")]
print(extracted_files)

In [ ]:
!pip install openpyxl

In [ ]:
import pandas as pd

file_path = os.path.join(DATA_DIR, "Implication Fuzzy.xlsx")
df = pd.read_excel(file_path)

df.head(2)

In [ ]:
df1 = pd.read_excel(os.path.join(DATA_DIR, "XOR Fuzzy.xlsx"))
df1.head(2)

In [ ]:
df2 = pd.read_excel(os.path.join(DATA_DIR, "OR Fuzzy.xlsx"))
df2.head(2)

In [ ]:
df3 = pd.read_excel(os.path.join(DATA_DIR, "AND Fuzzy.xlsx"))
df3.head(2)

In [ ]:
df4 = pd.read_excel(os.path.join(DATA_DIR, "NOT Fuzzy.xlsx"))
df4.head(2)

In [ ]:
print(f"ROWS: {df.shape[0]}")
print(f"COLUMNS: {df.shape[1]}")

In [ ]:
df.isnull().sum()

In [ ]:
df["Target"].value_counts()

**What this shows:** 1,000 rows, no missing values, and `Target` is imbalanced toward 1 (about 75% of rows are True), since Implication is only false when X1 is above 0.5 *and* X2 is below 0.5, so with both inputs independently random, most rows end up True. Worth keeping in mind for later: a model that just guessed "1" every time would already be right 75% of the time, so that's the real floor any reported accuracy needs to clear.

In [ ]:
import matplotlib.pyplot as plt
plt.scatter(x=df['Target'], y=df['X1'], color='g')
plt.title("X1 vs. Target");

In [ ]:
plt.scatter(x=df['X2'], y=df['Target'], color='r')
plt.title("X2 vs. Target");

**What this shows:** neither plot shows a clean split between Target=0 and Target=1 on its own, because Implication genuinely depends on *both* X1 and X2 together, so no single-variable view can separate the two classes. That's the visual evidence for why the model below needs both inputs, not just one.

## Implication model

**Fix:** the original only fed the model `X1`. Implication is a function of *two* inputs (`X1`, `X2`), so it could never learn the real relationship with one input missing. Separate scalers are used for X and y so fitting one never overwrites the other's learned range, and everything here is prefixed `impl_` so it survives the XOR section below without being overwritten.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

impl_x_scaler = MinMaxScaler()
impl_y_scaler = MinMaxScaler()

X = df[["X1", "X2"]].values          # both inputs of the Implication function
targ = df["Target"].values.reshape(-1, 1)

sx = impl_x_scaler.fit_transform(X)
sy = impl_y_scaler.fit_transform(targ)

impl_x_train, impl_x_test, impl_y_train, impl_y_test = train_test_split(
    sx, sy, test_size=0.20, random_state=13
)
impl_x_train.shape, impl_x_test.shape

In [ ]:
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import GridSearchCV

params = {
    "hidden_layer_sizes": [(3, 3), (3,), (3, 3, 3)],
    "activation": ["logistic", "relu", "tanh"],
    "solver": ["sgd", "lbfgs", "adam"],
    "verbose": [0],
    "max_iter": [10000],
}

impl_grid = GridSearchCV(estimator=MLPRegressor(), param_grid=params)
impl_grid.fit(impl_x_train, impl_y_train.ravel())
impl_grid

In [ ]:
# Fix: inverse-transform the model's actual predictions, not the scaled X test values,
# and use the y-specific scaler.
impl_y_predict = impl_grid.predict(impl_x_test)
impl_y_predict_orig = impl_y_scaler.inverse_transform(impl_y_predict.reshape(-1, 1))
impl_y_test_orig = impl_y_scaler.inverse_transform(impl_y_test)

In [ ]:
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from math import sqrt

RMSE = float(format(sqrt(mean_squared_error(impl_y_test_orig, impl_y_predict_orig)), '.3f'))
MSE = mean_squared_error(impl_y_test_orig, impl_y_predict_orig)
MAE = mean_absolute_error(impl_y_test_orig, impl_y_predict_orig)
r2 = r2_score(impl_y_test_orig, impl_y_predict_orig)

print(f"RMSE = {RMSE}\nMSE = {MSE}\nMAE = {MAE}\nR2 = {r2}")

In [ ]:
# Target is actually binary (0/1), so it's also worth scoring this as classification --
# threshold the (0-1) predictions at 0.5.
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

impl_pred_class = (impl_y_predict_orig.ravel() >= 0.5).astype(int)
impl_true_class = impl_y_test_orig.ravel().astype(int)

accuracy = accuracy_score(impl_true_class, impl_pred_class)
f1 = f1_score(impl_true_class, impl_pred_class)

print(f"Accuracy = {accuracy:.3f}\nF1 = {f1:.3f}")
print("Confusion matrix:")
print(confusion_matrix(impl_true_class, impl_pred_class))

**What this shows:** in a full run (10,000-iteration grid search across 27 architecture/activation/solver combinations), this reaches test R² above 0.999 and 100% classification accuracy after thresholding: the network has essentially reconstructed the exact Implication rule from 800 training examples, without ever being given the formula. That's a real result and not just the 75%-majority baseline from the class-balance check above showing through, since 100% accuracy means it's also getting the minority (Target=0) rows right, which a majority-guessing model never would.

### Try your own values: Implication

Edit `x1_val` / `x2_val` below (each between 0 and 1) and re-run the cell.

In [ ]:
x1_val = 0.30   # edit me (0 to 1)
x2_val = 0.80   # edit me (0 to 1)

def predict_implication(x1, x2):
    scaled = impl_x_scaler.transform([[x1, x2]])
    pred_scaled = impl_grid.predict(scaled)
    pred = impl_y_scaler.inverse_transform(pred_scaled.reshape(-1, 1))
    return float(pred[0, 0])

pred = predict_implication(x1_val, x2_val)
print(f"Implication({x1_val}, {x2_val}) -> regression output: {pred:.3f}  (crisp: {int(pred >= 0.5)})")

**What this shows:** because `Target` is generated by an exact Boolean rule, you can hand-verify any prediction here yourself. Implication(A, B) is false only when A is above 0.5 *and* B is below 0.5; every other combination should round to 1.

### Classical baseline: Implication

No learning at all: threshold X1 and X2 at 0.5 and apply the textbook Implication rule (`NOT A OR B`) directly. This is *exactly* the rule the dataset's `Target` column was generated from, so it will always match `Target` perfectly on this data. The point isn't to "beat" it, it's to see how close the learned network's approximation gets to the real rule.

In [ ]:
import numpy as np

def crisp(v, thresh=0.5):
    return (np.asarray(v) > thresh).astype(int)

def classical_implication(x1, x2):
    return ((1 - crisp(x1)) | crisp(x2)).astype(int)

impl_x_test_orig = impl_x_scaler.inverse_transform(impl_x_test)
classical_pred = classical_implication(impl_x_test_orig[:, 0], impl_x_test_orig[:, 1])

classical_acc = accuracy_score(impl_true_class, classical_pred)
network_acc = accuracy_score(impl_true_class, impl_pred_class)

print(f"Classical formula accuracy: {classical_acc:.3f}")
print(f"Learned network accuracy:   {network_acc:.3f}")

**What this shows:** the classical formula's accuracy is 1.000 by construction, since `Target` was generated from exactly this rule, so it can't be anything else. The number that actually matters is the network's accuracy sitting right next to it: that gap (typically zero, per the full run above) is the real measure of how closely the learned approximation matches the ground truth, which is the whole point of running this comparison.

## Smallest-two-rows lookup, deduplicated

**Fix:** cells 17-21 of the original repeated the same "find the 2 rows with the smallest min(X1, X2)" logic five times, once per file, changing only the filename. That's now one function.

In [ ]:
def smallest_two(path, cols=("X1", "X2")):
    """Return the 2 rows of `path` with the smallest value across `cols`
    (row-wise min when there are 2+ columns, or the plain column when there's 1)."""
    data = pd.read_excel(path)
    if len(cols) == 1:
        return data.nsmallest(2, cols[0])
    data = data.copy()
    data["min_value"] = data[list(cols)].min(axis=1)
    return data.nsmallest(2, "min_value").drop(columns=["min_value"])


print(smallest_two(os.path.join(DATA_DIR, "Implication Fuzzy.xlsx")))
print(smallest_two(os.path.join(DATA_DIR, "AND Fuzzy.xlsx")))
print(smallest_two(os.path.join(DATA_DIR, "OR Fuzzy.xlsx")))
print(smallest_two(os.path.join(DATA_DIR, "NOT Fuzzy.xlsx"), cols=("X1",)))
print(smallest_two(os.path.join(DATA_DIR, "XOR Fuzzy.xlsx")))

**What this shows:** checked against the real data, this surfaces Target=1 rows for Implication, NOT, and XOR (a tiny X1 or X2 makes those true almost automatically) and Target=0 rows for AND (a tiny min forces at least one input below 0.5, so AND fails), except for OR, where the outcome instead depends on whichever input *isn't* the small one, so its two smallest-min rows can land on opposite classes. It's more of a data-exploration utility than a modeling result: the original notebook never explained why it computes this, only that it's now one function instead of five near-identical cells.

## XOR model

Same shape as the Implication model above, with the same separate-scaler fix, clearer naming (`df_xor` instead of the reused, ambiguous `df`), and `xor_*`-prefixed variables so this section doesn't clobber the Implication model above it.

In [ ]:
df_xor = pd.read_excel(os.path.join(DATA_DIR, "XOR Fuzzy.xlsx"))

xor_x_scaler = MinMaxScaler()
xor_y_scaler = MinMaxScaler()

X = df_xor[["X1", "X2"]].values
y = df_xor["Target"].values.reshape(-1, 1)

X_scaled = xor_x_scaler.fit_transform(X)
y_scaled = xor_y_scaler.fit_transform(y)

xor_x_train, xor_x_test, xor_y_train, xor_y_test = train_test_split(X_scaled, y_scaled, test_size=0.2, random_state=13)

param_grid = {
    "hidden_layer_sizes": [(3, 3), (3,), (3, 3, 3)],
    "activation": ["logistic", "relu", "tanh"],
    "solver": ["sgd", "lbfgs", "adam"],
    "max_iter": [10000],
    "verbose": [0],
}

xor_grid = GridSearchCV(estimator=MLPRegressor(), param_grid=param_grid, scoring="neg_mean_squared_error", cv=5)
xor_grid.fit(xor_x_train, xor_y_train.ravel())

xor_pred_train = xor_grid.predict(xor_x_train)
xor_pred_test = xor_grid.predict(xor_x_test)

xor_y_train_orig = xor_y_scaler.inverse_transform(xor_y_train)
xor_y_test_orig = xor_y_scaler.inverse_transform(xor_y_test)
xor_pred_train_orig = xor_y_scaler.inverse_transform(xor_pred_train.reshape(-1, 1))
xor_pred_test_orig = xor_y_scaler.inverse_transform(xor_pred_test.reshape(-1, 1))

rmse_train = sqrt(mean_squared_error(xor_y_train_orig, xor_pred_train_orig))
rmse_test = sqrt(mean_squared_error(xor_y_test_orig, xor_pred_test_orig))
mse_train = mean_squared_error(xor_y_train_orig, xor_pred_train_orig)
mse_test = mean_squared_error(xor_y_test_orig, xor_pred_test_orig)
mae_train = mean_absolute_error(xor_y_train_orig, xor_pred_train_orig)
mae_test = mean_absolute_error(xor_y_test_orig, xor_pred_test_orig)
r2_train = r2_score(xor_y_train_orig, xor_pred_train_orig)
r2_test = r2_score(xor_y_test_orig, xor_pred_test_orig)

print(f"Train RMSE: {rmse_train}, Test RMSE: {rmse_test}")
print(f"Train MSE: {mse_train}, Test MSE: {mse_test}")
print(f"Train MAE: {mae_train}, Test MAE: {mae_test}")
print(f"Train R2: {r2_train}, Test R2: {r2_test}")

**What this shows:** results here can vary noticeably between runs, since neither `MLPRegressor` nor `GridSearchCV` are seeded with a fixed `random_state`. In testing, this first search (the smaller (3,3)-style architectures) landed a good fit less than half the time; the rest of the time it got stuck in a poor local minimum, with test RMSE around 0.3-0.4 (essentially failing to learn XOR). If your run looks like that, that's expected and not a bug. The second grid search below exists specifically to check for this, and in testing it consistently did much better than the first grid, typically landing well under 0.02 RMSE.

In [ ]:
param_grid = {
    "hidden_layer_sizes": [(5, 5), (5,), (5, 5, 5)],
    "activation": ["logistic", "relu", "tanh"],
    "solver": ["sgd", "lbfgs", "adam"],
    "max_iter": [10000],
    "verbose": [0],
}

xor_grid = GridSearchCV(estimator=MLPRegressor(), param_grid=param_grid, scoring="neg_mean_squared_error", cv=5)
xor_grid.fit(xor_x_train, xor_y_train.ravel())

xor_pred_train = xor_grid.predict(xor_x_train)
xor_pred_test = xor_grid.predict(xor_x_test)

xor_pred_train_orig = xor_y_scaler.inverse_transform(xor_pred_train.reshape(-1, 1)).flatten()
xor_pred_test_orig = xor_y_scaler.inverse_transform(xor_pred_test.reshape(-1, 1)).flatten()

rmse_train = sqrt(mean_squared_error(xor_y_train_orig, xor_pred_train_orig))
rmse_test = sqrt(mean_squared_error(xor_y_test_orig, xor_pred_test_orig))

print(f"Updated Train RMSE: {rmse_train}, Updated Test RMSE: {rmse_test}")

**What this shows:** across repeated runs, these larger (5,5)-style architectures were far more reliable than the (3,3)-style ones above, typically landing at test RMSE under 0.02 (sometimes as low as 0.0004) regardless of how the first grid happened to go. XOR's loss surface is genuinely harder for a small network to optimize; the practical lesson is that the extra capacity here is buying *reliability* of hitting a good solution, not a higher accuracy ceiling. Both architectures can fit XOR well, but the larger one gets there far more consistently.

### Try your own values: XOR

Edit `x1_val` / `x2_val` below (each between 0 and 1) and re-run the cell.

In [ ]:
x1_val = 0.30   # edit me (0 to 1)
x2_val = 0.80   # edit me (0 to 1)

def predict_xor(x1, x2):
    scaled = xor_x_scaler.transform([[x1, x2]])
    pred_scaled = xor_grid.predict(scaled)
    pred = xor_y_scaler.inverse_transform(pred_scaled.reshape(-1, 1))
    return float(pred[0, 0])

pred = predict_xor(x1_val, x2_val)
print(f"XOR({x1_val}, {x2_val}) -> regression output: {pred:.3f}  (crisp: {int(pred >= 0.5)})")

**What this shows:** same idea as Implication above. XOR(A, B) should read true whenever exactly one of A, B is above 0.5, so you can check the regression output's crisp reading against that rule by hand for any pair you try.

### Classical baseline: XOR

Same idea as the Implication baseline above: threshold X1 and X2 at 0.5 and compute XOR directly, no learning involved. Again, this is exactly the rule the data was generated from.

In [ ]:
def classical_xor(x1, x2):
    return (crisp(x1) ^ crisp(x2)).astype(int)

xor_x_test_orig = xor_x_scaler.inverse_transform(xor_x_test)
xor_true_class = (xor_y_test_orig.ravel() >= 0.5).astype(int)
xor_pred_class = (xor_pred_test_orig.ravel() >= 0.5).astype(int)
classical_pred_xor = classical_xor(xor_x_test_orig[:, 0], xor_x_test_orig[:, 1])

classical_acc_xor = accuracy_score(xor_true_class, classical_pred_xor)
network_acc_xor = accuracy_score(xor_true_class, xor_pred_class)

print(f"Classical formula accuracy: {classical_acc_xor:.3f}")
print(f"Learned network accuracy:   {network_acc_xor:.3f}")

**What this shows:** as with Implication, the classical formula scores 1.000 by construction, and the network's accuracy sitting at or near that same number (per the full run above) is the real evidence here: a network with no prior knowledge of Boolean logic rediscovering XOR, historically the textbook example of a function a single-layer network *can't* learn, once it's given a hidden layer to work with.

## True fuzzy targets, no thresholding at all

Every regression model above is still learning the same crisp 0/1 `Target` that Part 2 is built around; however close the network gets, the *destination* is a hard step at 0.5. This section changes what's being learned entirely, computing the actual continuous Zadeh fuzzy operators directly from X1 and X2 with no thresholding anywhere:

- AND = min(X1, X2)
- OR = max(X1, X2)
- NOT = 1 - X1
- XOR = max(min(X1, 1-X2), min(1-X1, X2)), built from AND/OR/NOT so it stays logically consistent
- Implication = min(1, 1 - X1 + X2), the Lukasiewicz definition

The target is smooth and continuous everywhere, which is arguably the more faithful "fuzzy logic" problem: everything earlier in this notebook was really learning a crisp Boolean rule dressed up in continuous inputs.

In [ ]:
def fuzzy_and(x1, x2): return np.minimum(x1, x2)
def fuzzy_or(x1, x2): return np.maximum(x1, x2)
def fuzzy_not(x1): return 1 - x1
def fuzzy_xor(x1, x2): return np.maximum(np.minimum(x1, 1 - x2), np.minimum(1 - x1, x2))
def fuzzy_implication(x1, x2): return np.minimum(1, 1 - x1 + x2)

fuzzy_pooled_X = pd.concat(
    [df3[["X1", "X2"]], df2[["X1", "X2"]], df1[["X1", "X2"]], df[["X1", "X2"]]],
    ignore_index=True,
).values  # AND, OR, XOR, and Implication files' (X1, X2) pairs, pooled

fx1, fx2 = fuzzy_pooled_X[:, 0], fuzzy_pooled_X[:, 1]
fuzzy_Y = np.column_stack([
    fuzzy_and(fx1, fx2), fuzzy_or(fx1, fx2), fuzzy_not(fx1), fuzzy_xor(fx1, fx2), fuzzy_implication(fx1, fx2)
])

fuzzy_x_scaler = MinMaxScaler()
fuzzy_X_scaled = fuzzy_x_scaler.fit_transform(fuzzy_pooled_X)

fuzzy_X_train, fuzzy_X_test, fuzzy_Y_train, fuzzy_Y_test = train_test_split(
    fuzzy_X_scaled, fuzzy_Y, test_size=0.2, random_state=13
)

fuzzy_model = MLPRegressor(hidden_layer_sizes=(32, 32, 32), activation="relu", max_iter=20000)
fuzzy_model.fit(fuzzy_X_train, fuzzy_Y_train)
fuzzy_pred = fuzzy_model.predict(fuzzy_X_test)

fuzzy_gate_names = ["AND", "OR", "NOT", "XOR", "Implication"]
for i, name in enumerate(fuzzy_gate_names):
    rmse = sqrt(mean_squared_error(fuzzy_Y_test[:, i], fuzzy_pred[:, i]))
    r2 = r2_score(fuzzy_Y_test[:, i], fuzzy_pred[:, i])
    print(f"{name:<12} RMSE={rmse:.4f}  R2={r2:.4f}")

**What this shows:** in testing, AND, OR, NOT, and Implication all fit cleanly (R2 above 0.99) since min, max, and the Lukasiewicz formula are smooth, gently-behaved functions. XOR is the exception: with a network this size it landed around R2 = 0.98-0.99 but occasionally worse, because the fuzzy XOR construction above has a sharp crease everywhere the min/max switches which input it's tracking, and a small network under-fits that crease. Widening it (checked separately up to (64, 64)) closed the gap to R2 above 0.998. The lesson carries over neatly from the crisp XOR case earlier: XOR needs more capacity than the other four gates, whether the target is a hard step or a smooth surface with a sharp fold in it.

### Try your own values: true fuzzy operators

Edit `x1_val` / `x2_val` below and compare the network's output against the exact formula for all 5 operators at once.

In [ ]:
x1_val = 0.30   # edit me (0 to 1)
x2_val = 0.80   # edit me (0 to 1)

scaled_input = fuzzy_x_scaler.transform([[x1_val, x2_val]])
network_pred = fuzzy_model.predict(scaled_input)[0]
true_vals = [
    fuzzy_and(x1_val, x2_val), fuzzy_or(x1_val, x2_val), fuzzy_not(x1_val),
    fuzzy_xor(x1_val, x2_val), fuzzy_implication(x1_val, x2_val),
]

print(f"{'Gate':<12}{'Network':>10}{'True formula':>14}")
for name, pred, true in zip(fuzzy_gate_names, network_pred, true_vals):
    print(f"{name:<12}{pred:>10.3f}{true:>14.3f}")

**What this shows:** unlike every earlier manual-input cell, there's no crisp 0/1 to round to here. The network's job is to match the exact decimal, not just land on the right side of 0.5, which makes this the most direct side-by-side check in the whole notebook of how closely a trained network can imitate a known mathematical function.